# Astronomical Object Classifier — Baseline (ResNet50)

End-to-end transfer-learning pipeline for classifying astronomical images into
five classes: spiral galaxy, elliptical galaxy, nebula, star cluster, planetary object.

**Approach.** ResNet50 pretrained on ImageNet. Two phases:

1. Freeze backbone, train new 5-class head for a few epochs.
2. Unfreeze everything, fine-tune end-to-end at low learning rate.

**Goal.** Establish a reproducible baseline at ≥ 80% test accuracy. Section 10
adds Grad-CAM for interpretability; sections 11–13 cover the EfficientNet/ViT
comparison, a class-weighting ablation, and out-of-distribution detection.

## 1. Setup and environment

In [ ]:
# Detect Colab vs local; expose IS_COLAB flag used later.
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print("Running in Colab" if IS_COLAB else "Running locally")

In [ ]:
# Install deps. On Colab this tops up the runtime; locally it only adds the
# light deps (your CUDA torch is already in the venv -- don't overwrite it with
# a CPU wheel).
if IS_COLAB:
    %pip install -q torch torchvision scikit-learn matplotlib seaborn pillow tqdm
else:
    %pip install -q scikit-learn matplotlib seaborn pillow tqdm ipywidgets

In [ ]:
# Standard imports.
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

# Reproducibility. Won't make CUDA fully deterministic, but enough for a baseline.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Device.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__}, device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠ No GPU detected. On Colab: Runtime → Change runtime type → T4 GPU.")
    print("  Training will work on CPU but will be ~30-50x slower.")

## 2. Configuration

Every knob in one cell. Edit here, not scattered through the notebook.

In [ ]:
CONFIG = {
    # Paths
    'data_dir':     '/content/drive/MyDrive/astro_classifier/data/processed' if IS_COLAB
                    else 'data/processed',
    'checkpoint_dir': '/content/drive/MyDrive/astro_classifier/checkpoints' if IS_COLAB
                    else 'checkpoints',
    'results_dir':  '/content/drive/MyDrive/astro_classifier/results' if IS_COLAB
                    else 'results',

    # Data
    'image_size':   224,
    'val_frac':     0.15,
    'test_frac':    0.15,

    # Training
    'batch_size':   32,
    # 0 on Windows (Jupyter DataLoader workers hang there); 2 on Colab/Linux.
    'num_workers':  0 if os.name == 'nt' else 2,
    'phase1_epochs': 3,     # frozen backbone, train head only
    'phase2_epochs': 12,    # full fine-tune
    'phase1_lr':    1e-3,
    'phase2_lr':    1e-4,
    'weight_decay': 1e-4,
    'use_amp':      True,   # mixed precision; ~2x speedup on T4, no-op on CPU

    # Misc
    'use_class_weights': True,
    'resume_phase2_from': None,  # set to a checkpoint path to resume phase 2
}

# Class order is fixed by ImageFolder's alphabetical scan, but listing it here
# makes label semantics explicit when reading code.
CLASSES = [
    'elliptical_galaxy',
    'nebula',
    'planetary_object',
    'spiral_galaxy',
    'star_cluster',
]

# Majority classes get subsampled to MAJORITY_CAP each so the class imbalance
# is a manageable ~2.7:1 instead of ~6.7:1. Minority classes (star_cluster,
# planetary) are left at full size.
MAJORITY_CLASSES = {'spiral_galaxy', 'elliptical_galaxy', 'nebula'}
MAJORITY_CAP = 600

os.makedirs(CONFIG['data_dir'], exist_ok=True)
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
os.makedirs(CONFIG['results_dir'], exist_ok=True)
print("Config loaded. Data dir:", CONFIG['data_dir'])

## 3. Data loading

`ImageFolder` reads from `<data_dir>/<class_name>/*.jpg` and assigns labels
alphabetically. We then stratify-split indices for train / val / test.

In [ ]:
# Two transform pipelines: heavy augmentation for train, deterministic resize for eval.
# Stronger train-time augmentation: vertical flip + full 180° rotation (space has no
# canonical "up"), heavier color jitter, and RandomErasing as a regularizer.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(CONFIG['image_size'], scale=(0.7, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=180),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(CONFIG['image_size']),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Load the real dataset from disk. ImageFolder scans subfolders alphabetically,
# which must match the CLASSES order defined in section 2.
base_dataset = ImageFolder(CONFIG['data_dir'], transform=eval_tf)
assert base_dataset.classes == CLASSES, (
    f"Folder order {base_dataset.classes} != CLASSES {CLASSES}. "
    f"Check that data_dir subfolders are named exactly: {CLASSES}"
)
print(f"Total images on disk: {len(base_dataset)}")
print(f"Classes: {base_dataset.classes}")
raw_counts = np.bincount([y for _, y in base_dataset.samples], minlength=len(CLASSES))
print(f"Per-class counts (raw): {dict(zip(base_dataset.classes, raw_counts.tolist()))}")

# --- Subsample majority classes ---
# Build a list of dataset indices, capping each majority class at MAJORITY_CAP.
rng = np.random.RandomState(SEED)
class_to_indices = {c: [] for c in range(len(CLASSES))}
for idx, (_, label) in enumerate(base_dataset.samples):
    class_to_indices[label].append(idx)

kept_indices = []
for label, idxs in class_to_indices.items():
    cls_name = CLASSES[label]
    if cls_name in MAJORITY_CLASSES and len(idxs) > MAJORITY_CAP:
        idxs = rng.choice(idxs, size=MAJORITY_CAP, replace=False).tolist()
    kept_indices.extend(idxs)

kept_indices = sorted(kept_indices)
kept_counts = np.bincount([base_dataset.samples[i][1] for i in kept_indices],
                          minlength=len(CLASSES))
print(f"\nPer-class counts (after subsampling): "
      f"{dict(zip(base_dataset.classes, kept_counts.tolist()))}")
print(f"Total after subsampling: {len(kept_indices)}")


In [ ]:
# Stratified split over the SUBSAMPLED indices.
kept_indices_arr = np.array(kept_indices)
kept_labels = np.array([base_dataset.samples[i][1] for i in kept_indices])

train_idx, temp_idx = train_test_split(
    kept_indices_arr, test_size=CONFIG['val_frac'] + CONFIG['test_frac'],
    stratify=kept_labels, random_state=SEED,
)
# Stratify the val/test split using the labels of temp_idx.
temp_labels = np.array([base_dataset.samples[i][1] for i in temp_idx])
val_rel = CONFIG['val_frac'] / (CONFIG['val_frac'] + CONFIG['test_frac'])
val_idx, test_idx = train_test_split(
    temp_idx, test_size=1 - val_rel,
    stratify=temp_labels, random_state=SEED,
)

print(f"train: {len(train_idx)}, val: {len(val_idx)}, test: {len(test_idx)}")
for name, idxs in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    c = np.bincount([base_dataset.samples[i][1] for i in idxs], minlength=len(CLASSES))
    print(f"  {name:<6} " + ", ".join(f"{CLASSES[k]}={c[k]}" for k in range(len(CLASSES))))


class TransformedSubset(torch.utils.data.Dataset):
    '''Wraps the base dataset, applying a per-split transform and reading from disk.'''
    def __init__(self, base, indices, transform):
        self.base = base
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        path, label = self.base.samples[self.indices[i]]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label


train_ds = TransformedSubset(base_dataset, train_idx, train_tf)
val_ds   = TransformedSubset(base_dataset, val_idx,   eval_tf)
test_ds  = TransformedSubset(base_dataset, test_idx,  eval_tf)

# Class-imbalance handling: weighted cross-entropy loss ONLY (built in section 6).
# After subsampling the imbalance is only ~2.7:1, so a WeightedRandomSampler on
# top of weighted CE would over-correct (it makes batches ~uniform, then the loss
# weights up the minority classes AGAIN) and tends to hurt minority-class
# precision. Plain shuffling + weighted CE is the right amount of correction here.
pin = (DEVICE.type == 'cuda')
train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True,
                          num_workers=CONFIG['num_workers'], pin_memory=pin)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'], shuffle=False,
                          num_workers=CONFIG['num_workers'], pin_memory=pin)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG['batch_size'], shuffle=False,
                          num_workers=CONFIG['num_workers'], pin_memory=pin)


In [ ]:
# Sanity-check: visualize a training batch (with augmentation applied).
def denormalize(tensor):
    # Undo ImageNet normalization so the augmented tensors display as real colors.
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1)


imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(13, 5.5))
for i, ax in enumerate(axes.flat):
    if i >= len(imgs):
        ax.axis('off')
        continue
    ax.imshow(denormalize(imgs[i]).permute(1, 2, 0))
    ax.set_title(base_dataset.classes[labels[i]], fontsize=9)
    ax.axis('off')
plt.suptitle('Training batch sample (augmentation applied)')
plt.tight_layout()
plt.show()

## 4. Model

ResNet50 pretrained on ImageNet. Replace the final 1000-class head with a
new linear layer for our 5 classes. The new head trains from scratch; the
backbone starts with ImageNet weights.

In [ ]:
# Transfer learning: start from an ImageNet-pretrained ResNet50 and swap only
# the classifier head (1000-way -> 5-way), reusing the learned visual features.
def build_model(num_classes=len(CLASSES)):
    weights = models.ResNet50_Weights.IMAGENET1K_V2
    model = models.resnet50(weights=weights)
    # Replace the final FC layer with a fresh 5-class head.
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model


# Phase-1 helper: train the head only by freezing everything except `fc`.
def freeze_backbone(model):
    for name, param in model.named_parameters():
        param.requires_grad = name.startswith('fc.')


# Phase-2 helper: unfreeze the whole network for end-to-end fine-tuning.
def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True


model = build_model().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"ResNet50 built. Parameters: {n_params / 1e6:.1f}M")

## 5. Training loop utilities

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    """One pass over `loader`. If `optimizer` is given, train; else evaluate."""
    # Passing/withholding an optimizer is what selects train vs. eval mode,
    # so the same function serves both loops.
    is_train = optimizer is not None
    model.train(is_train)
    use_amp = (scaler is not None) and (DEVICE.type == 'cuda')

    total_loss, total_correct, total = 0.0, 0, 0
    pbar = tqdm(loader, leave=False, desc='train' if is_train else 'eval')
    for imgs, labels in pbar:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        # Disable grad tracking during eval to save memory/time.
        with torch.set_grad_enabled(is_train):
            # AMP: run the forward pass in float16 for speed/lower VRAM on GPU.
            if use_amp:
                with torch.cuda.amp.autocast():
                    logits = model(imgs)
                    loss = criterion(logits, labels)
            else:
                logits = model(imgs)
                loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                if use_amp:
                    # Scale the loss before backward so small fp16 grads don't underflow.
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()

        # Weight by batch size so the final average is correct even if the
        # last batch is smaller.
        total_loss    += loss.item() * imgs.size(0)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total         += imgs.size(0)
        pbar.set_postfix(loss=f"{total_loss/total:.3f}",
                         acc=f"{total_correct/total:.3f}")

    return total_loss / total, total_correct / total


def train(model, train_loader, val_loader, criterion, optimizer, scheduler,
          epochs, phase_name, ckpt_path, start_epoch=1, best_val_acc=0.0,
          scaler=None):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    start = time.time()

    for epoch in range(start_epoch, epochs + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, scaler)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
        if scheduler is not None:
            scheduler.step()

        history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
        history['val_loss'].append(val_loss);  history['val_acc'].append(val_acc)

        # Checkpoint only when validation accuracy improves (early-stopping style:
        # we keep the best model, not the last one).
        improved = val_acc > best_val_acc
        if improved:
            best_val_acc = val_acc
            torch.save({
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict() if scheduler else None,
                'scaler_state': scaler.state_dict() if scaler else None,
                'classes': CLASSES,
                'val_acc': val_acc,
                'phase': phase_name,
                'epoch': epoch,
            }, ckpt_path)

        flag = '★' if improved else ' '
        print(f"[{phase_name}] epoch {epoch:2d}/{epochs} "
              f"train_loss={tr_loss:.3f} train_acc={tr_acc:.3f} "
              f"val_loss={val_loss:.3f} val_acc={val_acc:.3f} {flag}")

    print(f"{phase_name} done in {(time.time() - start)/60:.1f} min, "
          f"best val_acc={best_val_acc:.3f} -> {ckpt_path}")
    return history

In [ ]:
# Class weights for the loss, computed on the TRAINING split only so no
# information leaks from val/test.
if CONFIG['use_class_weights']:
    train_labels = np.array([base_dataset.samples[i][1] for i in train_idx])
    counts = np.bincount(train_labels, minlength=len(CLASSES))
    # Balanced weighting: rare classes get a higher weight so each class
    # contributes equally to the loss.
    weights = counts.sum() / (len(CLASSES) * counts)
    class_weights = torch.tensor(weights, dtype=torch.float, device=DEVICE)
    print("Class weights:", dict(zip(CLASSES, [f'{w:.3f}' for w in weights])))
else:
    class_weights = None

criterion = nn.CrossEntropyLoss(weight=class_weights)

## 6. Phase 1 — Train classification head only

Backbone frozen. Only the final linear layer learns. This converges quickly
and gives the head reasonable weights before we start unfreezing.

In [ ]:
# Phase 1: backbone frozen, so only the new head is trainable. A higher LR
# (1e-3) is safe here since we aren't disturbing the pretrained weights.
freeze_backbone(model)
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"Trainable parameters (phase 1): {sum(p.numel() for p in trainable):,}")

optimizer1 = torch.optim.AdamW(trainable, lr=CONFIG['phase1_lr'],
                               weight_decay=CONFIG['weight_decay'])
scaler1 = torch.cuda.amp.GradScaler() if (CONFIG['use_amp'] and DEVICE.type == 'cuda') else None
ckpt1 = os.path.join(CONFIG['checkpoint_dir'], 'phase1_best.pt')

history1 = train(model, train_loader, val_loader, criterion,
                 optimizer1, scheduler=None,
                 epochs=CONFIG['phase1_epochs'],
                 phase_name='phase1',
                 ckpt_path=ckpt1,
                 scaler=scaler1)

## 7. Phase 2 — Fine-tune full network

Load the best phase-1 head, unfreeze everything, train at a low learning rate
with cosine annealing.

In [ ]:
# Resume from best phase-1 checkpoint by default, OR from a phase-2 checkpoint
# if CONFIG['resume_phase2_from'] is set (handy when free-tier Colab disconnects).
resume_path = CONFIG.get('resume_phase2_from') or ckpt1
ckpt = torch.load(resume_path, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
unfreeze_all(model)

trainable = [p for p in model.parameters() if p.requires_grad]
print(f"Trainable parameters (phase 2): {sum(p.numel() for p in trainable):,}")
print(f"Resumed from: {resume_path} (val_acc={ckpt.get('val_acc', 0):.3f}, "
      f"epoch={ckpt.get('epoch', 0)}, phase={ckpt.get('phase', 'n/a')})")

# Phase 2: all layers trainable, so use a 10x smaller LR (1e-4) to avoid
# wrecking the pretrained features; cosine schedule decays it smoothly to ~0.
optimizer2 = torch.optim.AdamW(trainable, lr=CONFIG['phase2_lr'],
                               weight_decay=CONFIG['weight_decay'])
scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer2, T_max=CONFIG['phase2_epochs'])
scaler2 = torch.cuda.amp.GradScaler() if (CONFIG['use_amp'] and DEVICE.type == 'cuda') else None

# Restore optimizer / scheduler / scaler / start-epoch if we're resuming mid-phase-2.
start_epoch = 1
best_val_acc = 0.0
if ckpt.get('phase') == 'phase2':
    if ckpt.get('optimizer_state'): optimizer2.load_state_dict(ckpt['optimizer_state'])
    if ckpt.get('scheduler_state'): scheduler2.load_state_dict(ckpt['scheduler_state'])
    if ckpt.get('scaler_state') and scaler2: scaler2.load_state_dict(ckpt['scaler_state'])
    start_epoch = ckpt.get('epoch', 0) + 1
    best_val_acc = ckpt.get('val_acc', 0.0)
    print(f"Resuming phase 2 at epoch {start_epoch} (best val_acc so far: {best_val_acc:.3f})")

ckpt2 = os.path.join(CONFIG['checkpoint_dir'], 'phase2_best.pt')
history2 = train(model, train_loader, val_loader, criterion,
                 optimizer2, scheduler2,
                 epochs=CONFIG['phase2_epochs'],
                 phase_name='phase2',
                 ckpt_path=ckpt2,
                 start_epoch=start_epoch,
                 best_val_acc=best_val_acc,
                 scaler=scaler2)

In [ ]:
# Plot training curves.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
all_train_loss = history1['train_loss'] + history2['train_loss']
all_val_loss   = history1['val_loss']   + history2['val_loss']
all_train_acc  = history1['train_acc']  + history2['train_acc']
all_val_acc    = history1['val_acc']    + history2['val_acc']
boundary = len(history1['train_loss'])
xs = range(1, len(all_train_loss) + 1)

axes[0].plot(xs, all_train_loss, label='train')
axes[0].plot(xs, all_val_loss,   label='val')
axes[0].axvline(boundary + 0.5, color='gray', linestyle='--', alpha=0.6, label='unfreeze')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend()

axes[1].plot(xs, all_train_acc, label='train')
axes[1].plot(xs, all_val_acc,   label='val')
axes[1].axvline(boundary + 0.5, color='gray', linestyle='--', alpha=0.6)
axes[1].set_title('Accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend()
plt.tight_layout()
plt.savefig(f"{CONFIG['checkpoint_dir']}/fig_training_curves.png", dpi=150, bbox_inches='tight')
plt.show()

## 8. Test-set evaluation

In [ ]:
# Reload the BEST phase-2 weights (the final epoch isn't necessarily the best).
best = torch.load(ckpt2, map_location=DEVICE)
model.load_state_dict(best['model_state'])
model.eval()

all_preds, all_labels = [], []
# no_grad: we're only measuring, no backprop, so skip the autograd overhead.
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc='test'):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

acc = accuracy_score(all_labels, all_preds)
print(f"\nTest accuracy: {acc:.4f}")
print("\nPer-class report:")
print(classification_report(all_labels, all_preds,
                            target_names=base_dataset.classes, digits=3))

In [ ]:
# Confusion matrix.
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=base_dataset.classes, yticklabels=base_dataset.classes)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix (test)')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

## 9. Save final model

The best checkpoint is already saved to `phase2_best.pt`. The cell below
exports a clean, inference-ready bundle.

In [ ]:
# Final inference checkpoint. This dict schema is the CONTRACT the Streamlit
# demo loads against: weights + class order + the exact preprocessing
# (image_size, ImageNet mean/std) so inference preprocessing matches training.
final_path = os.path.join(CONFIG['checkpoint_dir'], 'baseline_resnet50.pt')
torch.save({
    'model_state':  model.state_dict(),
    'classes':      base_dataset.classes,
    'image_size':   CONFIG['image_size'],
    'imagenet_mean': IMAGENET_MEAN,
    'imagenet_std':  IMAGENET_STD,
    'test_accuracy': acc,
}, final_path)
print(f"Saved final baseline to {final_path}")

## 10. Grad-CAM — explaining what the model looked at

Grad-CAM produces a heatmap over the input image showing which regions most
influenced the predicted class. It works by:

1. Forward pass — record the activations `A` of the last convolutional layer
   (for ResNet50 this is `layer4`, the 7×7×2048 feature map before pooling).
2. Pick a target class `c`. Backprop the logit for class `c` and record the
   gradients flowing into `A`.
3. For each of the 2048 channels, average the gradient spatially to get a
   single importance weight `α_k`.
4. Take a weighted sum of the feature-map channels using these weights, apply
   ReLU to keep only positive contributions, and upsample to image size.

Result: a 7×7 heatmap interpolated up to 224×224, overlaid on the original
image. Where the heatmap is bright, that's where the model looked.

This implementation uses PyTorch *hooks* — callbacks that fire when a specific
layer runs its forward or backward pass — to grab the feature map and gradient
without modifying the model code.

In [ ]:
class GradCAM:
    '''Grad-CAM for a torch.nn.Module. Pass the target conv layer at construction.'''

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        # Register hooks. They fire every time `target_layer` runs.
        self._fwd = target_layer.register_forward_hook(self._save_activation)
        self._bwd = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        # grad_output[0] is dL/dA — what we need.
        self.gradients = grad_output[0].detach()

    def remove(self):
        '''Detach hooks. Call when done or the model will keep accumulating refs.'''
        self._fwd.remove()
        self._bwd.remove()

    def __call__(self, input_tensor, class_idx=None):
        '''
        input_tensor: (1, 3, H, W) normalized image on the correct device
        class_idx: which class to explain. If None, uses the predicted class.
        Returns: (heatmap as HxW numpy array in [0,1], predicted class idx, prob)
        '''
        self.model.eval()
        # Forward pass triggers _save_activation.
        logits = self.model(input_tensor)
        probs = torch.softmax(logits, dim=1)
        if class_idx is None:
            class_idx = int(logits.argmax(dim=1).item())

        # Backward pass triggers _save_gradient.
        self.model.zero_grad()
        score = logits[0, class_idx]
        score.backward(retain_graph=False)

        # activations: (1, C, h, w)   gradients: (1, C, h, w)
        # Channel weights = global-average-pool of gradients over (h, w).
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)        # (1, C, 1, 1)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)    # (1, 1, h, w)
        cam = torch.relu(cam)

        # Upsample to input resolution and normalize to [0, 1].
        cam = torch.nn.functional.interpolate(
            cam, size=input_tensor.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        prob = probs[0, class_idx].item()
        return cam, class_idx, prob


def overlay_heatmap(img_tensor, heatmap, alpha=0.45):
    '''Blend a [0,1] heatmap onto a normalized image tensor. Returns an HxWx3 numpy array.'''
    img = denormalize(img_tensor).permute(1, 2, 0).numpy()        # (H, W, 3) in [0,1]
    cmap = plt.get_cmap('jet')
    colored = cmap(heatmap)[..., :3]                              # drop alpha channel
    return np.clip((1 - alpha) * img + alpha * colored, 0, 1)

### Visualize Grad-CAM on test-set samples

We grab one image per class from the test set, run Grad-CAM, and display the
original alongside the overlay. Sanity check: the bright regions should land
on the actual object (spiral arms, planet disk, etc.) — not on the background
or image borders.

In [ ]:
# ResNet50's last conv block is `layer4`. For EfficientNet it would be `features[-1]`;
# for ViT, Grad-CAM doesn't apply directly (use attention maps instead).
cam_extractor = GradCAM(model, model.layer4)

# Pick one example per class from the test set.
class_to_test_idx = {c: [] for c in range(len(base_dataset.classes))}
for i in test_idx:
    _, label = base_dataset.samples[i]
    class_to_test_idx[label].append(i)

picks = [class_to_test_idx[c][0] for c in range(len(base_dataset.classes))
         if class_to_test_idx[c]]

fig, axes = plt.subplots(2, len(picks), figsize=(len(picks) * 2.6, 5.5))
for col, idx in enumerate(picks):
    path, true_label = base_dataset.samples[idx]
    img = Image.open(path).convert('RGB')
    img_tensor = eval_tf(img).unsqueeze(0).to(DEVICE)

    heatmap, pred_class, prob = cam_extractor(img_tensor)
    overlay = overlay_heatmap(img_tensor[0], heatmap)

    axes[0, col].imshow(denormalize(img_tensor[0]).permute(1, 2, 0))
    axes[0, col].set_title(f"true: {base_dataset.classes[true_label]}", fontsize=9)
    axes[0, col].axis('off')

    axes[1, col].imshow(overlay)
    pred_name = base_dataset.classes[pred_class]
    correct = '✓' if pred_class == true_label else '✗'
    axes[1, col].set_title(f"{correct} pred: {pred_name} ({prob:.2f})", fontsize=9)
    axes[1, col].axis('off')

plt.suptitle('Grad-CAM: original (top) vs heatmap overlay (bottom)')
plt.tight_layout()
plt.show()

# CRITICAL: remove the hooks when done. Otherwise the model keeps capturing
# activations on every forward pass, which leaks memory and slows training.
cam_extractor.remove()

In [ ]:
# ===========================================================================
# Grad-CAM ERROR ANALYSIS — the star_cluster -> nebula confusion
# ---------------------------------------------------------------------------
# Paste AFTER the existing Grad-CAM cells (after cell 34). Reuses the GradCAM
# class, overlay_heatmap(), and denormalize() already defined above.
#
# Goal: for each star_cluster that was misclassified as nebula, show three
# panels:
#   (1) the original image
#   (2) Grad-CAM explaining the model's WRONG choice (why "nebula"?)
#   (3) Grad-CAM explaining the CORRECT class (did the model even look at the
#       resolved stars when forced to justify "star_cluster"?)
#
# Interpretation guide (what to write in the report):
#   • If (2) lights up diffuse background / nebulosity AND (3) is weak or
#     scattered  -> the model is keying on diffuse glow, not resolved point
#     sources. This is the "embedded cluster in nebulosity" failure. More
#     sparse-open-cluster data or point-source-preserving augmentation could
#     help.
#   • If (3) clearly lights up the star field (the model CAN localize the
#     cluster) but it still predicts nebula  -> the features are present but
#     the decision boundary favors nebula. This points to class
#     prior / threshold / data-balance, not a perception failure.
#   • A correctly-classified star_cluster is shown last as a reference for
#     what "correct" attention looks like.
# ===========================================================================

# --- Find the specific error type: true=star_cluster, pred=nebula ----------
sc_idx  = base_dataset.classes.index('star_cluster')
neb_idx = base_dataset.classes.index('nebula')

# all_preds / all_labels were produced by the test-evaluation cell (section 9),
# aligned positionally with test_idx.
sc_to_neb_errors = [
    test_idx[i] for i in range(len(test_idx))
    if all_labels[i] == sc_idx and all_preds[i] == neb_idx
]

# A correctly-classified star_cluster, for reference.
sc_correct = [
    test_idx[i] for i in range(len(test_idx))
    if all_labels[i] == sc_idx and all_preds[i] == sc_idx
]

print(f"star_cluster -> nebula misclassifications: {len(sc_to_neb_errors)}")
print(f"correctly classified star_clusters available for reference: {len(sc_correct)}")

cam_extractor = GradCAM(model, model.layer4)

# Build the panel list: all the errors, plus one correct reference at the end.
panel_indices = list(sc_to_neb_errors)
ref_index = sc_correct[0] if sc_correct else None

n_rows = len(panel_indices) + (1 if ref_index is not None else 0)
if n_rows == 0:
    print("No star_cluster -> nebula errors on this run. "
          "Nothing to analyze (that's a good sign!).")
else:
    fig, axes = plt.subplots(n_rows, 3, figsize=(9, 3 * n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, 3)

    def render_row(row, idx, is_reference=False):
        path, true_label = base_dataset.samples[idx]
        img = Image.open(path).convert('RGB')
        img_tensor = eval_tf(img).unsqueeze(0).to(DEVICE)

        # Panel 1: original
        axes[row, 0].imshow(denormalize(img_tensor[0]).permute(1, 2, 0))
        tag = "REFERENCE (correct)" if is_reference else "ERROR"
        axes[row, 0].set_title(f"{tag}\ntrue: star_cluster", fontsize=9)
        axes[row, 0].axis('off')

        # Panel 2: Grad-CAM for the predicted (or, for reference, the same) class
        hm_pred, pred_class, prob_pred = cam_extractor(img_tensor)  # predicted class
        axes[row, 1].imshow(overlay_heatmap(img_tensor[0], hm_pred))
        pred_name = base_dataset.classes[pred_class]
        mark = '✓' if pred_class == true_label else '✗'
        axes[row, 1].set_title(
            f"{mark} CAM for prediction:\n{pred_name} ({prob_pred:.2f})",
            fontsize=9, color=('green' if mark == '✓' else 'darkred'))
        axes[row, 1].axis('off')

        # Panel 3: Grad-CAM forced to the star_cluster class
        hm_sc, _, prob_sc = cam_extractor(img_tensor, class_idx=sc_idx)
        axes[row, 2].imshow(overlay_heatmap(img_tensor[0], hm_sc))
        axes[row, 2].set_title(
            f"CAM for star_cluster\n(p={prob_sc:.2f})", fontsize=9)
        axes[row, 2].axis('off')

    row = 0
    for idx in panel_indices:
        render_row(row, idx, is_reference=False)
        row += 1
    if ref_index is not None:
        render_row(row, ref_index, is_reference=True)

    plt.suptitle(
        "star_cluster → nebula errors: what did the model attend to?\n"
        "Col 2 = why it chose nebula | Col 3 = where it sees 'star_cluster'",
        fontsize=11)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(f"{CONFIG['results_dir']}/fig_gradcam_correct.png", dpi=150, bbox_inches='tight')
    plt.show()

cam_extractor.remove()  # always detach hooks

### What to look for

- **Bright regions on the object itself** — model is doing what you want.
- **Bright regions on background, borders, or watermarks** — the model has
  learned a shortcut. Common culprits: telescope vignette patterns, label
  text overlaid on press-release images, JPEG compression artifacts at the
  edges.
- **Heatmap concentrated on one tiny spot** — model may be over-relying on a
  single cue (e.g. a bright central star in a galaxy). Often harmless but
  worth noting in your report.
- **Diffuse heatmap covering the whole frame** — model isn't very confident in
  *where* the evidence is. Often appears for misclassified samples.

For your evaluation report, include 5-10 Grad-CAM panels: a couple correct
predictions across different classes, plus every confusion type from the
confusion matrix. The misclassifications are often more informative than the
correct ones — they show you exactly which feature led the model astray.

In [ ]:
# Optional: Grad-CAM on a misclassified example, which is usually more diagnostic.
wrong_indices = [test_idx[i] for i in range(len(test_idx))
                 if all_preds[i] != all_labels[i]]

if not wrong_indices:
    print("No misclassifications on the test set — nothing to visualize here.")
else:
    cam_extractor = GradCAM(model, model.layer4)
    n_show = min(4, len(wrong_indices))
    fig, axes = plt.subplots(2, n_show, figsize=(n_show * 2.6, 5.5))
    if n_show == 1:
        axes = axes.reshape(2, 1)

    for col, idx in enumerate(wrong_indices[:n_show]):
        path, true_label = base_dataset.samples[idx]
        img = Image.open(path).convert('RGB')
        img_tensor = eval_tf(img).unsqueeze(0).to(DEVICE)

        # Explain the model's (wrong) prediction.
        heatmap, pred_class, prob = cam_extractor(img_tensor)
        overlay = overlay_heatmap(img_tensor[0], heatmap)

        axes[0, col].imshow(denormalize(img_tensor[0]).permute(1, 2, 0))
        axes[0, col].set_title(f"true: {base_dataset.classes[true_label]}", fontsize=9)
        axes[0, col].axis('off')

        axes[1, col].imshow(overlay)
        axes[1, col].set_title(
            f"✗ pred: {base_dataset.classes[pred_class]} ({prob:.2f})",
            fontsize=9, color='darkred')
        axes[1, col].axis('off')

    plt.suptitle('Grad-CAM on misclassified samples — what fooled the model?')
    plt.tight_layout()
    plt.savefig(f"{CONFIG['results_dir']}/fig_gradcam_errors.png", dpi=150, bbox_inches='tight')
    plt.show()
    cam_extractor.remove()

### 11.0 Preflight check

The comparison reuses objects built in sections 1–10 (the dataloaders,
`criterion`, the `train()` function, `base_dataset`, `CONFIG`, `CLASSES`). This
cell fails fast with a clear message if any of them are missing, instead of a
confusing `NameError` several cells later. It also verifies the label order and
that the ResNet50 baseline checkpoint exists.

In [ ]:
# Verify upstream state from baseline_resnet50 is in memory.
_required = ['CONFIG','CLASSES','DEVICE','train','train_loader','val_loader',
             'test_loader','criterion','base_dataset','models','nn','torch']
_missing = [n for n in _required if n not in globals()]
assert not _missing, (
    'Missing upstream objects: ' + ', '.join(_missing) +
    '\nRun sections 1-6 of baseline_resnet50 first (setup -> the '
    '"## 4. Data loading" ImageFolder cells -> model defs -> training fn -> '
    'criterion). Those ImageFolder cells are what build base_dataset and the '
    'loaders the comparison needs.'
)

# Label order MUST match the trained model or every metric is silently mislabeled.
assert list(base_dataset.classes) == list(CLASSES), (
    f'base_dataset.classes {base_dataset.classes} != CLASSES {CLASSES}. '
    'Class folders must be the long-form alphabetical names.'
)

# The ResNet50 baseline checkpoint must exist for the table to include ResNet50.
import os as _os
_resnet_ckpt = _os.path.join(CONFIG['checkpoint_dir'], 'phase2_best.pt')
print('Upstream state OK. base_dataset.classes =', base_dataset.classes)
print('ResNet50 baseline checkpoint present:', _os.path.exists(_resnet_ckpt))
print('  ->', _resnet_ckpt)
if not _os.path.exists(_resnet_ckpt):
    print('  WARNING: not found. Re-run phase-2 training (section 7), or point '
          "CONFIG['checkpoint_dir'] at the Drive folder that holds it, "
          'or ResNet50 will be skipped in the comparison.')

## 11. Architecture comparison

With the ResNet50 baseline established, EfficientNet-B0 and ViT-B/16 are trained
under the **same training protocol** and compared head-to-head.

The shared protocol is what makes the comparison fair: identical data splits,
augmentations, epoch counts, and optimizer config. Only the backbone changes.

This section reuses `CONFIG`, the dataloaders, the `GradCAM` class, and the
helper functions defined in sections 1–10.

### 11.1 Architecture factory

In [ ]:
# Architecture factory. Each builder swaps in a fresh 5-class head and returns
# (model, head_attr_name, gradcam_target_layer) -- the head name and the
# Grad-CAM target layer differ per architecture, so we capture them here.
def build_resnet50(num_classes=len(CLASSES)):
    weights = models.ResNet50_Weights.IMAGENET1K_V2
    m = models.resnet50(weights=weights)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m, 'fc', m.layer4  # head_attr_name, gradcam_target_layer


def build_efficientnet_b0(num_classes=len(CLASSES)):
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
    m = models.efficientnet_b0(weights=weights)
    # EfficientNet's classifier is a Sequential: [Dropout, Linear]
    in_features = m.classifier[1].in_features
    m.classifier[1] = nn.Linear(in_features, num_classes)
    return m, 'classifier', m.features[-1]


def build_vit_b16(num_classes=len(CLASSES)):
    weights = models.ViT_B_16_Weights.IMAGENET1K_V1
    m = models.vit_b_16(weights=weights)
    in_features = m.heads.head.in_features
    m.heads.head = nn.Linear(in_features, num_classes)
    # ViT doesn't support Grad-CAM the same way (no conv feature maps).
    # We return None for the grad-cam target and handle this in the eval cell.
    return m, 'heads', None


# Helper: freeze everything except the new head, by attribute name.
def freeze_except_head(model, head_attr_name):
    for name, param in model.named_parameters():
        param.requires_grad = name.startswith(head_attr_name + '.')


def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True


ARCHITECTURES = {
    'resnet50':        build_resnet50,
    'efficientnet_b0': build_efficientnet_b0,
    'vit_b_16':        build_vit_b16,
}
print(f"Available architectures: {list(ARCHITECTURES.keys())}")

### 11.2 Unified training function

Replicates the two-phase training logic from sections 6–7, parameterized by
architecture. The hyperparameters match the ResNet50 baseline so the comparison
is apples-to-apples.

In [ ]:
def train_architecture(arch_name, builder_fn, train_loader, val_loader,
                       criterion, ckpt_dir):
    """Run the full two-phase transfer-learning recipe for one backbone and
    return (best_phase2_checkpoint_path, param_count). Same protocol for every
    architecture, so the comparison stays fair."""
    print(f"\n{'='*60}\nTraining: {arch_name}\n{'='*60}")
    # builder_fn returns (model, head_attr_name, gradcam_layer); we only need
    # the model and the head name to know what to freeze in phase 1.
    model, head_attr, _ = builder_fn()
    model = model.to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {n_params / 1e6:.1f}M")

    # ---- Phase 1: frozen backbone, train the new head only (higher LR) ----
    freeze_except_head(model, head_attr)
    trainable = [p for p in model.parameters() if p.requires_grad]
    print(f"Phase 1 trainable: {sum(p.numel() for p in trainable):,}")

    opt1 = torch.optim.AdamW(trainable, lr=CONFIG['phase1_lr'],
                              weight_decay=CONFIG['weight_decay'])
    scaler1 = torch.cuda.amp.GradScaler() if (CONFIG['use_amp'] and DEVICE.type == 'cuda') else None
    ckpt1 = os.path.join(ckpt_dir, f'{arch_name}_phase1_best.pt')

    train(model, train_loader, val_loader, criterion, opt1, scheduler=None,
          epochs=CONFIG['phase1_epochs'], phase_name=f'{arch_name}-p1',
          ckpt_path=ckpt1, scaler=scaler1)

    # ---- Phase 2: reload the best head, unfreeze everything, fine-tune ----
    # Start phase 2 from the best phase-1 head, not wherever the last epoch left off.
    ckpt = torch.load(ckpt1, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    unfreeze_all(model)
    trainable = [p for p in model.parameters() if p.requires_grad]
    print(f"Phase 2 trainable: {sum(p.numel() for p in trainable):,}")

    # Lower LR (1e-4) + cosine decay so fine-tuning doesn't wreck the pretrained
    # features.
    opt2 = torch.optim.AdamW(trainable, lr=CONFIG['phase2_lr'],
                              weight_decay=CONFIG['weight_decay'])
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=CONFIG['phase2_epochs'])
    scaler2 = torch.cuda.amp.GradScaler() if (CONFIG['use_amp'] and DEVICE.type == 'cuda') else None
    ckpt2 = os.path.join(ckpt_dir, f'{arch_name}_phase2_best.pt')

    train(model, train_loader, val_loader, criterion, opt2, sch2,
          epochs=CONFIG['phase2_epochs'], phase_name=f'{arch_name}-p2',
          ckpt_path=ckpt2, scaler=scaler2)

    return ckpt2, n_params

### 11.3 Train EfficientNet-B0

ResNet50 does not need to be retrained here — its checkpoint already exists at
`checkpoints/phase2_best.pt`. The cell below copies it to the naming convention
the comparison table expects (`resnet50_phase2_best.pt`).

In [ ]:
# The comparison step expects each arch's checkpoint named '<arch>_phase2_best.pt'.
# Reuse the already-trained ResNet50 (saved as 'phase2_best.pt') by copying it to
# the expected name instead of retraining it.
import shutil
existing_resnet = os.path.join(CONFIG['checkpoint_dir'], 'phase2_best.pt')
target_resnet = os.path.join(CONFIG['checkpoint_dir'], 'resnet50_phase2_best.pt')
if os.path.exists(existing_resnet) and not os.path.exists(target_resnet):
    shutil.copy(existing_resnet, target_resnet)
    print(f"Copied existing ResNet50 checkpoint to {target_resnet}")

In [ ]:
# Train EfficientNet-B0 (~15-20 min on T4)
eff_ckpt, eff_params = train_architecture(
    'efficientnet_b0', build_efficientnet_b0,
    train_loader, val_loader, criterion,
    CONFIG['checkpoint_dir'],
)

### 11.4 Train ViT-B/16

ViT-B/16 has 86M parameters versus ResNet50's 25M and is more memory-hungry. If
training hits an out-of-memory error, two options:

1. Reduce the batch size to 16 (`CONFIG['batch_size']`).
2. Switch to a smaller ViT. torchvision does not ship ViT-S/16; it is available
   through the `timm` library:

   ```
   pip install -q timm
   import timm
   m = timm.create_model('vit_small_patch16_224', pretrained=True, num_classes=5)
   ```

In [ ]:
# Free up GPU memory from the previous run before training ViT
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Train ViT-B/16 (~20-30 min on T4 at batch size 32; reduce to 16 if OOM)
try:
    vit_ckpt, vit_params = train_architecture(
        'vit_b_16', build_vit_b16,
        train_loader, val_loader, criterion,
        CONFIG['checkpoint_dir'],
    )
except torch.cuda.OutOfMemoryError:
    print("\n⚠ Out of memory at batch size {}. Reduce batch size and rerun.".format(CONFIG['batch_size']))
    print("  In the CONFIG cell, set 'batch_size': 16, then re-run dataloader cell + this cell.")
    vit_ckpt, vit_params = None, None

### 11.5 Evaluate all architectures and build the comparison table

Loads each best checkpoint, runs it on the test set, measures inference latency,
and collects all metrics into a single DataFrame.

In [ ]:
import time
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report


def evaluate_architecture(arch_name, builder_fn, ckpt_path, test_loader, device):
    if ckpt_path is None or not os.path.exists(ckpt_path):
        return None
    model, _, _ = builder_fn()
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state['model_state'])
    model.to(device).eval()

    # Test-set accuracy + F1
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(test_loader, desc=f'eval {arch_name}', leave=False):
            imgs = imgs.to(device)
            preds = model(imgs).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    # macro_f1 averages per-class F1 equally, so minority classes count as much
    # as majority ones -- the honest metric under class imbalance.
    macro_f1 = f1_score(all_labels, all_preds, average='macro')

    # Inference latency: one forward pass on a single image, averaged over 50 runs
    dummy = torch.randn(1, 3, CONFIG['image_size'], CONFIG['image_size']).to(device)
    # Warmup
    with torch.no_grad():
        for _ in range(5):
            _ = model(dummy)
    # CUDA kernels are async; synchronize so the timer measures real GPU work.
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        for _ in range(50):
            _ = model(dummy)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    gpu_latency_ms = (time.time() - t0) / 50 * 1000

    # CPU latency too
    model_cpu = model.cpu()
    dummy_cpu = dummy.cpu()
    with torch.no_grad():
        for _ in range(3):
            _ = model_cpu(dummy_cpu)
    t0 = time.time()
    with torch.no_grad():
        for _ in range(10):
            _ = model_cpu(dummy_cpu)
    cpu_latency_ms = (time.time() - t0) / 10 * 1000

    n_params = sum(p.numel() for p in model.parameters())

    # Move back to device for any later use
    model.to(device)
 
    return {
        'architecture': arch_name,
        'parameters_M': round(n_params / 1e6, 1),
        'test_accuracy': round(acc, 4),
        'macro_f1': round(macro_f1, 4),
        'gpu_latency_ms': round(gpu_latency_ms, 2),
        'cpu_latency_ms': round(cpu_latency_ms, 1),
        'preds': all_preds,
        'labels': all_labels,
    }


# Run evaluation on all three architectures
results = []
for arch_name, builder_fn in ARCHITECTURES.items():
    ckpt = os.path.join(CONFIG['checkpoint_dir'], f'{arch_name}_phase2_best.pt')
    r = evaluate_architecture(arch_name, builder_fn, ckpt, test_loader, DEVICE)
    if r:
        results.append(r)
        print(f"{arch_name:20s}  acc={r['test_accuracy']:.3f}  f1={r['macro_f1']:.3f}  "
              f"gpu={r['gpu_latency_ms']:.1f}ms  cpu={r['cpu_latency_ms']:.0f}ms")

# Build the table
df = pd.DataFrame([{k: v for k, v in r.items() if k not in ['preds' , 'labels']}
                   for r in results])
print("\nComparison table:")
print(df.to_string(index=False))

# Save for the report
df.to_csv(os.path.join(CONFIG['checkpoint_dir'], 'comparison_table.csv'), index=False)
print(f"\nSaved comparison_table.csv to {CONFIG['checkpoint_dir']}")

### 11.6 Confusion matrix for each architecture

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4.5))
if len(results) == 1:
    axes = [axes]
for ax, r in zip(axes, results):
    cm = confusion_matrix(r['labels'], r['preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=base_dataset.classes, yticklabels=base_dataset.classes,
                ax=ax, cbar=False)
    ax.set_title(f"{r['architecture']}\nacc={r['test_accuracy']:.3f}, f1={r['macro_f1']:.3f}")
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['results_dir'], 'comparison_confusion_matrices.png'),
            dpi=100, bbox_inches='tight')
plt.show()

## 12. Ablation study — with vs without class weighting

This ablation justifies the class-weighting design choice with a number. The
best architecture from section 11 is retrained with `use_class_weights = False`,
and per-class F1 is compared against the weighted version.

In [ ]:
# Build an unweighted criterion (no class weights)
unweighted_criterion = nn.CrossEntropyLoss()

# Pick the best architecture from section 12 (the one with highest test accuracy)
best_arch = max(results, key=lambda r: r['test_accuracy'])['architecture']
print(f"Running ablation on the best architecture: {best_arch}")
builder = ARCHITECTURES[best_arch]

# Train without class weights
ablation_ckpt, _ = train_architecture(
    f'{best_arch}_no_weights', builder,
    train_loader, val_loader, unweighted_criterion,
    CONFIG['checkpoint_dir'],
)

In [ ]:
# Evaluate the ablation model and compare per-class F1 against the weighted version
ablation_result = evaluate_architecture(
    f'{best_arch}_no_weights', builder, ablation_ckpt, test_loader, DEVICE,
)
baseline_result = next(r for r in results if r['architecture'] == best_arch)

# Per-class F1 comparison
from sklearn.metrics import f1_score
weighted_f1   = f1_score(baseline_result['labels'], baseline_result['preds'],
                          average=None)
unweighted_f1 = f1_score(ablation_result['labels'], ablation_result['preds'],
                          average=None)

ablation_df = pd.DataFrame({
    'class': base_dataset.classes,
    'F1 (with weights)':    [round(f, 3) for f in weighted_f1],
    'F1 (without weights)': [round(f, 3) for f in unweighted_f1],
    'delta': [round(w - u, 3) for w, u in zip(weighted_f1, unweighted_f1)],
})
print(ablation_df.to_string(index=False))
ablation_df.to_csv(os.path.join(CONFIG['checkpoint_dir'], 'ablation_class_weights.csv'),
                   index=False)
print(f"\nSaved ablation_class_weights.csv")

## 13. Out-of-distribution (OOD) detection

The model so far always predicts one of five classes with confident
probabilities, even for inputs that belong to none. This section adds OOD
detection: a score that flags inputs that do not look like the training data.

Two methods are implemented, both requiring **zero additional training**:

- **MSP (Maximum Softmax Probability)** — the simplest baseline. The OOD score
  is `1 - max(softmax(logits))`. In-distribution images produce peaky
  probability distributions; OOD images produce diffuse ones.
- **Energy-based scoring** — `-logsumexp(logits)`. Beats MSP on most
  benchmarks. Same model, same forward pass, different post-processing.

The result is a clean separation between in-distribution and out-of-distribution
score distributions, measured by AUROC.

### 13.1 Load the OOD module and best model

In [ ]:
# Drop ood_detection.py in the same directory as this notebook before running.
from ood_detection import (
    score_image, score_batch, calibrate_thresholds, compute_auroc,
    msp_score, energy_score,
)

# Load the best architecture from section 12. If you skipped section 12, use
# the ResNet50 baseline from section 8.
best_arch_for_ood = best_arch if 'best_arch' in dir() else 'resnet50'
best_builder = ARCHITECTURES[best_arch_for_ood]
best_ckpt = os.path.join(CONFIG['checkpoint_dir'], f'{best_arch_for_ood}_phase2_best.pt')

ood_model, _, _ = best_builder()
ood_model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE)['model_state'])
ood_model.to(DEVICE).eval()
print(f"Using {best_arch_for_ood} for OOD scoring")

### 13.2 Score the in-distribution test set

Establishing how the model behaves on in-distribution images is a prerequisite
for defining what looks "unusual." The test set is run through both OOD scorers.

In [ ]:
# Score every image in the test set
in_dist_images = []
in_dist_labels = []
for idx in test_idx:
    path, label = base_dataset.samples[idx]
    in_dist_images.append(Image.open(path).convert('RGB'))
    in_dist_labels.append(label)

print(f"Scoring {len(in_dist_images)} in-distribution test images...")
in_dist_scores = score_batch(in_dist_images, ood_model, eval_tf, DEVICE)
print(f"MSP scores:    mean={in_dist_scores['msp_scores'].mean():.3f}, "
      f"95th pct={np.percentile(in_dist_scores['msp_scores'], 95):.3f}")
print(f"Energy scores: mean={in_dist_scores['energy_scores'].mean():.3f}, "
      f"95th pct={np.percentile(in_dist_scores['energy_scores'], 95):.3f}")

### 13.3 Collect out-of-distribution images

The OOD set needs images that are clearly not astronomical content. CIFAR-10
(10 classes of natural images) is used here via torchvision as the most
reproducible choice — roughly a 170 MB download on first run. ImageNet samples
or arbitrary photographs would serve equally well as OOD sources.

In [ ]:
from torchvision.datasets import CIFAR10
from torchvision import transforms as T

# Download CIFAR-10 test set (10,000 images, 32x32). We'll upscale a sample of
# them to 224x224 for the model.
cifar_path = os.path.join(CONFIG['checkpoint_dir'], '..', 'cifar10')
os.makedirs(cifar_path, exist_ok=True)
cifar = CIFAR10(root=cifar_path, train=False, download=True,
                 transform=T.Resize(224))  # 32 -> 224, intentionally blurry

# Sample 100 OOD images
np.random.seed(SEED)
ood_indices = np.random.choice(len(cifar), size=100, replace=False)
ood_images = [cifar[i][0] for i in ood_indices]  # PIL.Image, already resized

# Score them
print(f"Scoring {len(ood_images)} out-of-distribution CIFAR-10 images...")
ood_scores = score_batch(ood_images, ood_model, eval_tf, DEVICE)
print(f"MSP scores:    mean={ood_scores['msp_scores'].mean():.3f}")
print(f"Energy scores: mean={ood_scores['energy_scores'].mean():.3f}")

### 13.4 Visualize the score distributions

The key plot: do the in-distribution and OOD score distributions visibly
separate?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# MSP histogram
axes[0].hist(in_dist_scores['msp_scores'], bins=30, alpha=0.6, label='In-distribution (astro)', color='steelblue')
axes[0].hist(ood_scores['msp_scores'],    bins=30, alpha=0.6, label='OOD (CIFAR-10)',         color='salmon')
axes[0].set_xlabel('MSP score (1 - max softmax prob)')
axes[0].set_ylabel('count')
axes[0].set_title('MSP — Maximum Softmax Probability')
axes[0].legend()

# Energy histogram
axes[1].hist(in_dist_scores['energy_scores'], bins=30, alpha=0.6, label='In-distribution (astro)', color='steelblue')
axes[1].hist(ood_scores['energy_scores'],    bins=30, alpha=0.6, label='OOD (CIFAR-10)',         color='salmon')
axes[1].set_xlabel('energy score (-logsumexp of logits)')
axes[1].set_ylabel('count')
axes[1].set_title('Energy-based scoring')
axes[1].legend()

plt.tight_layout()
out_path = os.path.join(CONFIG['checkpoint_dir'], 'ood_score_distributions.png')
plt.savefig(out_path, dpi=100, bbox_inches='tight')
plt.show()
print(f"Saved {out_path}")

### 13.5 Compute AUROC

A single summary number: how well does each score separate in-distribution from
OOD? AUROC of 1.0 is perfect separation, 0.5 is random.

In [ ]:
# AUROC = how well a score separates in-dist from OOD across all thresholds
# (1.0 = perfect separation, 0.5 = no better than chance).
msp_auroc = compute_auroc(in_dist_scores['msp_scores'],
                           ood_scores['msp_scores'])
energy_auroc = compute_auroc(in_dist_scores['energy_scores'],
                              ood_scores['energy_scores'])

print(f"MSP AUROC:    {msp_auroc:.4f}")
print(f"Energy AUROC: {energy_auroc:.4f}")
print(f"\nBetter method: {'Energy' if energy_auroc > msp_auroc else 'MSP'} "
      f"(by {abs(energy_auroc - msp_auroc):.3f})")

# Save for the report
ood_summary = {
    'msp_auroc': float(msp_auroc),
    'energy_auroc': float(energy_auroc),
    'in_dist_n': len(in_dist_images),
    'ood_n': len(ood_images),
    'best_arch': best_arch_for_ood,
}
import json as _json
with open(os.path.join(CONFIG['checkpoint_dir'], 'ood_summary.json'), 'w') as f:
    _json.dump(ood_summary, f, indent=2)

### 13.6 Calibrate operating threshold

Flagging OOD images in the Streamlit demo requires a concrete threshold. The
convention used here is "95% TPR": the threshold is chosen to keep 95% of
in-distribution images correctly classified as in-distribution, and the
resulting OOD detection rate is the operating point.

In [ ]:
thresholds = calibrate_thresholds(in_dist_scores, target_tpr=0.95)
print(f"Calibrated thresholds at 95% TPR:")
print(f"  MSP threshold:    {thresholds['msp_threshold']:.3f}")
print(f"  Energy threshold: {thresholds['energy_threshold']:.3f}")

# At these thresholds, what fraction of OOD images are correctly flagged?
msp_ood_caught    = (ood_scores['msp_scores'] > thresholds['msp_threshold']).mean()
energy_ood_caught = (ood_scores['energy_scores'] > thresholds['energy_threshold']).mean()
print(f"\nAt 95% in-dist TPR, OOD detection rate:")
print(f"  MSP:    {msp_ood_caught:.1%} of CIFAR-10 images flagged as OOD")
print(f"  Energy: {energy_ood_caught:.1%} of CIFAR-10 images flagged as OOD")

# Save thresholds for the demo to use
with open(os.path.join(CONFIG['checkpoint_dir'], 'ood_thresholds.json'), 'w') as f:
    _json.dump(thresholds, f, indent=2)
print(f"\nSaved ood_thresholds.json")

### 13.7 Test on a specific image

Sanity check: one in-distribution image and one OOD image are scored, confirming
the OOD scorer responds correctly to each.

In [ ]:
# Pick one of each
test_in_dist = in_dist_images[0]
test_ood = ood_images[0]

result_in  = score_image(test_in_dist, ood_model, eval_tf, DEVICE,
                          msp_threshold=thresholds['msp_threshold'],
                          energy_threshold=thresholds['energy_threshold'])
result_ood = score_image(test_ood, ood_model, eval_tf, DEVICE,
                          msp_threshold=thresholds['msp_threshold'],
                          energy_threshold=thresholds['energy_threshold'])

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(test_in_dist); axes[0].axis('off')
axes[0].set_title(f"In-dist: MSP={result_in.msp_score:.3f}, energy={result_in.energy_score:.2f}\n"
                  f"Flagged OOD: MSP={result_in.msp_is_ood}, energy={result_in.energy_is_ood}",
                  fontsize=9)
axes[1].imshow(test_ood); axes[1].axis('off')
axes[1].set_title(f"OOD: MSP={result_ood.msp_score:.3f}, energy={result_ood.energy_score:.2f}\n"
                  f"Flagged OOD: MSP={result_ood.msp_is_ood}, energy={result_ood.energy_is_ood}",
                  fontsize=9)
plt.tight_layout()
plt.show()

## 14. Summary and conclusions

This notebook built and evaluated a five-class astronomical image classifier
(spiral galaxy, elliptical galaxy, nebula, star cluster, planetary object) end
to end.

- **Baseline (sections 1–9).** ResNet50 pretrained on ImageNet, fine-tuned in
  two phases — a frozen-backbone head warmup, then full fine-tuning at a 10×
  lower learning rate with cosine annealing. Class imbalance was handled by
  capping the majority classes and weighting the cross-entropy loss. The model
  is evaluated on a held-out test set (target ≥ 80% accuracy) with a per-class
  classification report and a confusion matrix.

- **Interpretability (section 10).** Grad-CAM heatmaps verify the model attends
  to the object itself rather than background, borders, or watermark artifacts.
  A focused error analysis of the `star_cluster` → `nebula` confusion shows
  where the model's attention drifts toward diffuse nebulosity instead of
  resolved point sources.

- **Architecture comparison (section 11).** EfficientNet-B0 and ViT-B/16 were
  trained under an identical protocol and compared on accuracy, macro-F1,
  parameter count, and GPU/CPU latency, making the accuracy-vs-efficiency
  trade-off explicit.

- **Ablation (section 12).** Retraining the best architecture without class
  weights isolates the effect of the weighting choice on per-class F1,
  especially for the minority classes.

- **Out-of-distribution detection (section 13).** MSP and energy scores
  separate in-distribution astronomical images from CIFAR-10 (measured by
  AUROC), and a calibrated 95%-TPR threshold is exported for the Streamlit demo
  to flag non-astronomical uploads.

**Key takeaways.** Transfer learning with a two-phase schedule reaches a strong
baseline on a small, imbalanced dataset; Grad-CAM gives confidence that the
predictions rest on real morphology; and lightweight OOD scoring adds a safety
net at inference time with no extra training.

**Limitations and next steps.** The dataset is modest in size and partly sourced
from press-release imagery, which can introduce label-correlated artifacts;
collecting more raw survey data and adding point-source-preserving augmentation
would likely improve the `star_cluster` / `nebula` boundary. The exported
checkpoint (`baseline_resnet50.pt`) and `ood_thresholds.json` are the artifacts
consumed by the Streamlit demo.